# LeetCode 1452 — People Whose List of Favorite Companies Is Not a Subset of Another List bằng Python

Link đề: https://leetcode.com/problems/people-whose-list-of-favorite-companies-is-not-a-subset-of-another-list/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, String

Skeleton LeetCode:

```python
class Solution:
    def peopleIndexes(self, favoriteCompanies: list[list[str]]) -> list[int]:
        ...
```

## 1. Mục tiêu học

1. Hiểu quan hệ **tập con** (`A ⊆ B`) và dùng `set` của Python: `issubset`, toán tử `<=`, `<`.
2. Thấy vì sao kiểm tra tập con bằng **list** (`x in list`) chậm `O(m²)` còn bằng **set** chỉ `O(m)`.
3. **Mã hoá chuỗi thành số** (đánh id cho tên công ty) rồi biểu diễn mỗi danh sách bằng **bitmask** (số nguyên lớn của Python): `A ⊆ B ⇔ A & B == A`.
4. Tối ưu bằng **sắp xếp theo kích thước giảm dần** + tính **bắc cầu** của quan hệ tập con, và bằng **chỉ mục ngược** (company → tập người).
5. Tránh lỗi: so với chính mình, chỉ so một chiều, đảo chiều phép AND, so chuỗi ghép (substring), quên trả về chỉ số tăng dần.

## 2. Đề bài & ví dụ

Cho mảng `favoriteCompanies`, trong đó `favoriteCompanies[i]` là danh sách công ty yêu thích của người thứ `i` (đánh số từ 0).

Trả về **chỉ số** những người mà danh sách của họ **không phải tập con** của danh sách của **bất kỳ người nào khác**. Kết quả phải **tăng dần**.

**Ví dụ 1:** `[["leetcode","google","facebook"],["google","microsoft"],["google","facebook"],["google"],["amazon"]]` → `[0,1,4]`. Người 2 ⊆ người 0; người 3 ⊆ người 0 (và ⊆ người 1).

**Ví dụ 2:** `[["leetcode","google","facebook"],["leetcode","amazon"],["facebook","google"]]` → `[0,1]`.

**Ví dụ 3:** `[["leetcode"],["google"],["facebook"],["amazon"]]` → `[0,1,2,3]`.

**Ràng buộc:**
- `1 <= favoriteCompanies.length <= 100`, `1 <= favoriteCompanies[i].length <= 500`, `1 <= favoriteCompanies[i][j].length <= 20`.
- Các chuỗi trong cùng một danh sách **phân biệt**; các danh sách **khác nhau từng đôi** (sau khi sắp xếp).
- Chỉ gồm chữ thường tiếng Anh.

Hệ quả quan trọng: vì hai danh sách luôn khác nhau, `A ⊆ B` (với `A ≠ B`) kéo theo `|A| < |B|` — **tập con thực sự**.

In [1]:
import random
import string
import time
from collections import defaultdict
from typing import List

ex1 = [["leetcode", "google", "facebook"], ["google", "microsoft"], ["google", "facebook"], ["google"], ["amazon"]]
ex2 = [["leetcode", "google", "facebook"], ["leetcode", "amazon"], ["facebook", "google"]]
ex3 = [["leetcode"], ["google"], ["facebook"], ["amazon"]]
examples = [(ex1, [0, 1, 4]), (ex2, [0, 1]), (ex3, [0, 1, 2, 3])]
for fav, want in examples:
    print(len(fav), "người -> kỳ vọng", want)

5 người -> kỳ vọng [0, 1, 4]
3 người -> kỳ vọng [0, 1]
4 người -> kỳ vọng [0, 1, 2, 3]


## 3. Kiến thức nền

### 3.1 Tập con và `set`

`A ⊆ B` nếu mọi phần tử của `A` đều thuộc `B`. Trong Python: `A.issubset(B)` hoặc `A <= B`; `A < B` là **tập con thực sự** (`A ⊆ B` và `A ≠ B`). Kiểm tra `x in set` trung bình `O(1)`, nên `A <= B` tốn `O(|A|)`.

In [2]:
A = {"google", "facebook"}
B = {"leetcode", "google", "facebook"}
print(A.issubset(B), A <= B, A < B, B <= A, A <= A, A < A)

True True True False True False


### 3.2 `x in list` vs `x in set`

`in` trên list duyệt tuần tự `O(m)` ⇒ kiểm tra tập con bằng list tốn `O(m²)`. Với `m = 500` và `100·99` cặp người, chênh lệch rất lớn.

In [3]:
big_list = [f"c{i}" for i in range(500)]
big_set = set(big_list)
t = time.perf_counter(); r1 = all(x in big_list for x in big_list); t1 = time.perf_counter() - t
t = time.perf_counter(); r2 = all(x in big_set for x in big_list); t2 = time.perf_counter() - t
print(f"list: {t1*1000:.2f} ms | set: {t2*1000:.3f} ms | cùng kết quả {r1 == r2}")

list: 1.12 ms | set: 0.070 ms | cùng kết quả True


### 3.3 Đánh id cho chuỗi bằng dict

Gán mỗi tên công ty một số nguyên `0, 1, 2, …` theo lần đầu xuất hiện: `ids.setdefault(name, len(ids))`. Làm việc với số nhanh và gọn hơn chuỗi.

In [4]:
ids = {}
for lst in ex1:
    for c in lst:
        ids.setdefault(c, len(ids))
print(ids)

{'leetcode': 0, 'google': 1, 'facebook': 2, 'microsoft': 3, 'amazon': 4}


### 3.4 Bitmask bằng số nguyên lớn của Python

Mỗi danh sách ↦ số nguyên có bit thứ `id(c)` bật: `mask |= 1 << id`. Python có số nguyên **không giới hạn độ lớn**, nên vài chục nghìn bit vẫn được. Phép `&` thực hiện trên cả khối bit cùng lúc (rất nhanh).

`A ⊆ B ⇔ A & B == A` (mọi bit của A đều có trong B).

In [5]:
def to_mask(lst, ids):
    m = 0
    for c in lst:
        m |= 1 << ids[c]
    return m

masks = [to_mask(lst, ids) for lst in ex1]
for i, m in enumerate(masks):
    print(i, format(m, "05b")[::-1], "(bit 0 bên trái)", ex1[i])
print("người 2 ⊆ người 0?", masks[2] & masks[0] == masks[2])
print("người 1 ⊆ người 0?", masks[1] & masks[0] == masks[1])

0 11100 (bit 0 bên trái) ['leetcode', 'google', 'facebook']
1 01010 (bit 0 bên trái) ['google', 'microsoft']
2 01100 (bit 0 bên trái) ['google', 'facebook']
3 01000 (bit 0 bên trái) ['google']
4 00001 (bit 0 bên trái) ['amazon']
người 2 ⊆ người 0? True
người 1 ⊆ người 0? False


### 3.5 `sorted` theo khoá và `enumerate`

`sorted(range(n), key=lambda i: -len(fav[i]))` cho thứ tự chỉ số theo kích thước giảm dần — dùng trong cách tối ưu. Kết quả cuối phải `sorted(...)` lại để tăng dần.

In [6]:
order = sorted(range(len(ex1)), key=lambda i: -len(ex1[i]))
print("thứ tự theo kích thước giảm dần:", order, [len(ex1[i]) for i in order])

thứ tự theo kích thước giảm dần: [0, 1, 2, 3, 4] [3, 2, 2, 1, 1]


## 4. Ý tưởng nền tảng / trực giác

Với mỗi người `i`, hỏi: "có người `j ≠ i` nào mà danh sách `i` nằm trọn trong danh sách `j` không?" Nếu **không có** ⇒ giữ `i`.

Tính tay ví dụ 1 (L = leetcode, G = google, F = facebook, M = microsoft, A = amazon):

| i | danh sách | ⊆ người nào? | giữ? |
|---|---|---|---|
| 0 | {L,G,F} | không ai (ai cũng ít hơn 3 phần tử hoặc thiếu L) | ✔ |
| 1 | {G,M} | không ai có M | ✔ |
| 2 | {G,F} | ⊆ người 0 | ✘ |
| 3 | {G} | ⊆ người 0, 1, 2 | ✘ |
| 4 | {A} | không ai có A | ✔ |

Kết quả `[0,1,4]`.

In [7]:
sets = [set(l) for l in ex1]
for i, s in enumerate(sets):
    sup = [j for j, t in enumerate(sets) if j != i and s <= t]
    print(f"người {i} {sorted(s)} ⊆ {sup if sup else 'không ai'} -> {'giữ' if not sup else 'loại'}")

người 0 ['facebook', 'google', 'leetcode'] ⊆ không ai -> giữ
người 1 ['google', 'microsoft'] ⊆ không ai -> giữ
người 2 ['facebook', 'google'] ⊆ [0] -> loại
người 3 ['google'] ⊆ [0, 1, 2] -> loại
người 4 ['amazon'] ⊆ không ai -> giữ


## 5. Cách 1 — Brute Force / baseline (so mọi cặp bằng list)

**Ý tưởng:** với mỗi cặp `(i, j)`, `i ≠ j`, kiểm tra `all(c in fav[j] for c in fav[i])` — `in` trên **list**.

In [8]:
def people_brute(fav: List[List[str]]) -> List[int]:
    n = len(fav)
    res = []
    for i in range(n):
        is_subset = False
        for j in range(n):
            if i != j and all(c in fav[j] for c in fav[i]):
                is_subset = True
                break
        if not is_subset:
            res.append(i)
    return res

for fav, want in examples:
    print(people_brute(fav), "kỳ vọng", want)

[0, 1, 4] kỳ vọng [0, 1, 4]
[0, 1] kỳ vọng [0, 1]
[0, 1, 2, 3] kỳ vọng [0, 1, 2, 3]


**Độ phức tạp:** `n²` cặp × `m` phần tử × `in` list `O(m)` ⇒ `O(n²·m²)` phép so sánh chuỗi; với `n = 100`, `m = 500` là ~2,5·10⁹ — quá chậm.

**Nút thắt:** phép `in` trên list. Đổi sang `set` ⇒ `O(1)`.

In [9]:
def random_case(n, m, vocab, rng):
    # n danh sách phân biệt, mỗi danh sách 1..m công ty phân biệt lấy từ vocab
    seen, fav = set(), []
    while len(fav) < n:
        k = rng.randint(1, min(m, len(vocab)))
        lst = rng.sample(vocab, k)
        key = frozenset(lst)
        if key not in seen:
            seen.add(key)
            fav.append(lst)
    return fav

rng = random.Random(0)
vocab200 = [f"co{i}" for i in range(200)]
mid = random_case(40, 150, vocab200, rng)
t = time.perf_counter()
print(people_brute(mid), f"brute n=40, m<=150: {time.perf_counter()-t:.2f}s")

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 37, 38, 39] brute n=40, m<=150: 0.00s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — set:** đổi mỗi danh sách thành `set` một lần (`O(Σm)`), rồi `s_i <= s_j` tốn `O(|s_i|)` ⇒ `O(n²·m)`.

**Insight 2 — bitmask:** đánh id cho công ty, mỗi danh sách là một số nguyên; `mask_i & mask_j == mask_i` là phép trên số lớn, chạy ở tầng C theo từng khối 30–64 bit ⇒ rất nhanh.

**Insight 3 — sắp theo kích thước + bắc cầu:** tập con thực sự luôn **nhỏ hơn hẳn**. Duyệt người theo kích thước **giảm dần**; người `i` chỉ cần so với những người **đã được giữ** (lớn hơn). Vì nếu `i ⊆ j` mà `j` bị loại (`j ⊆ k`) thì `i ⊆ k` (bắc cầu), và có thể chọn `k` là một người được giữ.

**Insight 4 — chỉ mục ngược:** với mỗi công ty `c`, lưu tập người thích `c` dưới dạng bitmask `P[c]` (n ≤ 100 bit). Người `i` là tập con của ai đó ⇔ `AND_{c ∈ fav[i]} P[c]` còn bit nào **khác bit `i`** ⇒ `O(Σm)` phép AND trên số 100 bit.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — So mọi cặp bằng `set.issubset`

**Giải thích:** dựng `sets = [set(l) for l in fav]`; người `i` bị loại nếu `any(sets[i] <= sets[j] for j != i)`. Có thể lọc nhanh: chỉ so khi `len(sets[i]) < len(sets[j])`.

In [10]:
def people_sets(fav: List[List[str]]) -> List[int]:
    sets = [set(l) for l in fav]
    n = len(sets)
    return [i for i in range(n)
            if not any(i != j and len(sets[i]) < len(sets[j]) and sets[i] <= sets[j] for j in range(n))]

for fav, want in examples:
    print(people_sets(fav), "kỳ vọng", want)

[0, 1, 4] kỳ vọng [0, 1, 4]
[0, 1] kỳ vọng [0, 1]
[0, 1, 2, 3] kỳ vọng [0, 1, 2, 3]


**Code nộp LeetCode (cách 2):**

In [11]:
class SolutionSets:
    def peopleIndexes(self, favoriteCompanies: List[List[str]]) -> List[int]:
        sets = [set(l) for l in favoriteCompanies]
        res = []
        for i, s in enumerate(sets):
            if not any(i != j and s.issubset(t) for j, t in enumerate(sets)):
                res.append(i)
        return res

print(SolutionSets().peopleIndexes(ex1))

[0, 1, 4]


**Độ phức tạp:** Time `O(Σm + n²·m)` (≈ 5·10⁶ phép băm với giới hạn đề), Space `O(Σm)`.

### 7.2 Cách 3 — Mã hoá công ty thành id + bitmask

**Giải thích từng đoạn:**
1. `ids`: dict tên → số.
2. `masks[i]` = OR các `1 << ids[c]`.
3. Người `i` bị loại nếu có `j ≠ i` với `masks[i] & masks[j] == masks[i]`.

In [12]:
def people_bitmask(fav: List[List[str]]) -> List[int]:
    ids = {}
    masks = []
    for lst in fav:
        m = 0
        for c in lst:
            m |= 1 << ids.setdefault(c, len(ids))
        masks.append(m)
    n = len(masks)
    return [i for i in range(n)
            if not any(i != j and masks[i] & masks[j] == masks[i] for j in range(n))]

for fav, want in examples:
    print(people_bitmask(fav), "kỳ vọng", want)

[0, 1, 4] kỳ vọng [0, 1, 4]
[0, 1] kỳ vọng [0, 1]
[0, 1, 2, 3] kỳ vọng [0, 1, 2, 3]


**Code nộp LeetCode (cách 3):**

In [13]:
class SolutionBitmask:
    def peopleIndexes(self, favoriteCompanies: List[List[str]]) -> List[int]:
        ids, masks = {}, []
        for lst in favoriteCompanies:
            m = 0
            for c in lst:
                m |= 1 << ids.setdefault(c, len(ids))
            masks.append(m)
        return [i for i, a in enumerate(masks)
                if all(i == j or a & b != a for j, b in enumerate(masks))]

print(SolutionBitmask().peopleIndexes(ex2))

[0, 1]


**Độ phức tạp:** gọi `U` = số công ty khác nhau (≤ 50 000). Mỗi phép `&` tốn `O(U/w)` (w ≈ 30 bit/khối) ⇒ Time `O(Σm + n²·U/w)`, Space `O(n·U/w)`. Lưu ý: `m |= 1 << k` tạo số mới mỗi lần — với nhiều bit có thể dựng mask nhanh hơn bằng `sum(1 << k for k in set_ids)` hoặc `int` từ chuỗi bit, nhưng với giới hạn đề không cần.

### 7.3 Cách 4 — Sắp theo kích thước giảm dần, chỉ so với người được giữ

**Giải thích từng đoạn:**
1. `order` = chỉ số theo `len` giảm dần.
2. `kept` = danh sách người đã giữ (set của họ).
3. Người `i` được giữ nếu **không** là tập con của ai trong `kept` (những người trong `kept` đều có kích thước ≥; bằng nhau thì không thể là tập con vì danh sách khác nhau).
4. Trả về `sorted(...)` các chỉ số được giữ.

In [14]:
def people_sorted_kept(fav: List[List[str]]) -> List[int]:
    sets = [set(l) for l in fav]
    order = sorted(range(len(fav)), key=lambda i: -len(sets[i]))
    kept = []
    for i in order:
        if not any(sets[i] <= sets[k] for k in kept):
            kept.append(i)
    return sorted(kept)

class SolutionSortedKept:
    def peopleIndexes(self, favoriteCompanies: List[List[str]]) -> List[int]:
        sets = [set(l) for l in favoriteCompanies]
        kept = []
        for i in sorted(range(len(sets)), key=lambda i: -len(sets[i])):
            if not any(sets[i] <= sets[k] for k in kept):
                kept.append(i)
        return sorted(kept)

for fav, want in examples:
    print(people_sorted_kept(fav), SolutionSortedKept().peopleIndexes(fav), "kỳ vọng", want)

[0, 1, 4] [0, 1, 4] kỳ vọng [0, 1, 4]
[0, 1] [0, 1] kỳ vọng [0, 1]
[0, 1, 2, 3] [0, 1, 2, 3] kỳ vọng [0, 1, 2, 3]


**Độ phức tạp:** xấu nhất vẫn `O(n²·m)` (khi mọi người đều được giữ), nhưng thường ít phép so hơn nhiều. Space `O(Σm)`.

### 7.4 Cách 5 — Chỉ mục ngược: công ty → bitmask người

**Giải thích từng đoạn:**
1. `P[c]` = bitmask những người thích công ty `c`.
2. Với người `i`: `cand = AND của P[c]` với mọi `c` trong `fav[i]` = tập người chứa **toàn bộ** danh sách của `i` (luôn có chính `i`).
3. Bỏ bit `i`: nếu `cand & ~(1 << i)` khác 0 thì `i` là tập con của người khác ⇒ loại. Có thể dừng sớm khi `cand` chỉ còn bit `i`.

In [15]:
def people_inverted(fav: List[List[str]]) -> List[int]:
    P = defaultdict(int)
    for i, lst in enumerate(fav):
        for c in lst:
            P[c] |= 1 << i
    res = []
    for i, lst in enumerate(fav):
        me = 1 << i
        cand = ~0                         # "tất cả mọi người"
        for c in lst:
            cand &= P[c]
            if cand == me:                # chỉ còn chính mình -> chắc chắn giữ
                break
        if cand == me:
            res.append(i)
    return res

for fav, want in examples:
    print(people_inverted(fav), "kỳ vọng", want)

[0, 1, 4] kỳ vọng [0, 1, 4]
[0, 1] kỳ vọng [0, 1]
[0, 1, 2, 3] kỳ vọng [0, 1, 2, 3]


`~0` là `-1` (mọi bit đều bật trong biểu diễn bù hai vô hạn của Python), AND với `P[c]` (số dương) cho ra số dương. **Độ phức tạp:** Time `O(Σm · n/w)` — gần tuyến tính theo tổng kích thước input; Space `O(U · n/w)`.

So sánh tốc độ ở **giới hạn đề** (`n = 100`, mỗi danh sách tới 500 công ty):

In [16]:
rng = random.Random(1452)
vocab = [f"c{i}" for i in range(800)]
worst = random_case(100, 500, vocab, rng)
# thêm vài tập con thật sự để có người bị loại
for k in range(10):
    worst[k] = rng.sample(worst[k + 50], max(1, len(worst[k + 50]) // 2))
print("tổng số công ty trong input:", sum(map(len, worst)))
for name, f in [("sets", people_sets), ("bitmask", people_bitmask),
                ("sorted+kept", people_sorted_kept), ("inverted", people_inverted)]:
    t = time.perf_counter()
    r = f(worst)
    print(f"{name:12} -> {len(r):3d} người giữ ({time.perf_counter()-t:.4f}s)")

tổng số công ty trong input: 23681
sets         ->  89 người giữ (0.0033s)
bitmask      ->  89 người giữ (0.0069s)
sorted+kept  ->  89 người giữ (0.0015s)
inverted     ->  89 người giữ (0.0025s)


## 8. TRACE từng bước

**Trace cách 4** (sắp giảm dần + so với người được giữ) trên ví dụ 1. Kích thước: `[3,2,2,1,1]` ⇒ thứ tự `0,1,2,3,4` (sort ổn định giữ thứ tự khi hoà).

| bước | người i | danh sách | so với kept | kết quả | kept |
|---|---|---|---|---|---|
| 1 | 0 | {L,G,F} | (rỗng) | giữ | [0] |
| 2 | 1 | {G,M} | ⊄ 0 | giữ | [0,1] |
| 3 | 2 | {G,F} | ⊆ 0 | loại | [0,1] |
| 4 | 3 | {G} | ⊆ 0 | loại | [0,1] |
| 5 | 4 | {A} | ⊄ 0, ⊄ 1 | giữ | [0,1,4] |

In [17]:
def trace_sorted_kept(fav):
    sets = [set(l) for l in fav]
    order = sorted(range(len(fav)), key=lambda i: -len(sets[i]))
    print("thứ tự xét:", order)
    kept = []
    for step, i in enumerate(order, 1):
        checks = [(k, sets[i] <= sets[k]) for k in kept]
        sup = next((k for k, ok in checks if ok), None)
        if sup is None:
            kept.append(i)
        print(f"bước {step}: người {i} {sorted(sets[i])} | so với {[k for k, _ in checks]}"
              f" -> {'giữ' if sup is None else f'loại (⊆ người {sup})'} | kept={kept}")
    return sorted(kept)

print(trace_sorted_kept(ex1))

thứ tự xét: [0, 1, 2, 3, 4]
bước 1: người 0 ['facebook', 'google', 'leetcode'] | so với [] -> giữ | kept=[0]
bước 2: người 1 ['google', 'microsoft'] | so với [0] -> giữ | kept=[0, 1]
bước 3: người 2 ['facebook', 'google'] | so với [0, 1] -> loại (⊆ người 0) | kept=[0, 1]
bước 4: người 3 ['google'] | so với [0, 1] -> loại (⊆ người 0) | kept=[0, 1]
bước 5: người 4 ['amazon'] | so với [0, 1] -> giữ | kept=[0, 1, 4]
[0, 1, 4]


**Trace cách 5** (chỉ mục ngược) trên ví dụ 1 — `cand` in dạng tập người:

In [18]:
def bits(x, n):
    return [i for i in range(n) if x >> i & 1]

def trace_inverted(fav):
    n = len(fav)
    P = defaultdict(int)
    for i, lst in enumerate(fav):
        for c in lst:
            P[c] |= 1 << i
    print({c: bits(m, n) for c, m in P.items()})
    res = []
    for i, lst in enumerate(fav):
        cand = ~0
        for c in lst:
            cand &= P[c]
            print(f"  người {i}: AND với P[{c!r}] -> ứng viên chứa trọn {bits(cand, n)}")
        keep = cand == 1 << i
        print(f"người {i}: {'giữ' if keep else 'loại'}")
        if keep:
            res.append(i)
    return res

print(trace_inverted(ex1))

{'leetcode': [0], 'google': [0, 1, 2, 3], 'facebook': [0, 2], 'microsoft': [1], 'amazon': [4]}
  người 0: AND với P['leetcode'] -> ứng viên chứa trọn [0]
  người 0: AND với P['google'] -> ứng viên chứa trọn [0]
  người 0: AND với P['facebook'] -> ứng viên chứa trọn [0]
người 0: giữ
  người 1: AND với P['google'] -> ứng viên chứa trọn [0, 1, 2, 3]
  người 1: AND với P['microsoft'] -> ứng viên chứa trọn [1]
người 1: giữ
  người 2: AND với P['google'] -> ứng viên chứa trọn [0, 1, 2, 3]
  người 2: AND với P['facebook'] -> ứng viên chứa trọn [0, 2]
người 2: loại
  người 3: AND với P['google'] -> ứng viên chứa trọn [0, 1, 2, 3]
người 3: loại
  người 4: AND với P['amazon'] -> ứng viên chứa trọn [4]
người 4: giữ
[0, 1, 4]


## 9. Code nộp LeetCode

Lời giải chính: **set + sắp kích thước giảm dần + chỉ so với người được giữ** (cách 4) — dễ giải thích, nhanh trong thực tế, đúng chữ ký `peopleIndexes(self, favoriteCompanies)`.

In [19]:
from typing import List

class Solution:
    def peopleIndexes(self, favoriteCompanies: List[List[str]]) -> List[int]:
        sets = [set(lst) for lst in favoriteCompanies]
        order = sorted(range(len(sets)), key=lambda i: -len(sets[i]))
        kept = []
        for i in order:
            if not any(sets[i] <= sets[k] for k in kept):
                kept.append(i)
        return sorted(kept)

for fav, want in examples:
    assert Solution().peopleIndexes(fav) == want
print("Solution: 3 ví dụ đề OK")

Solution: 3 ví dụ đề OK


## 10. Vì sao thuật toán đúng

**Cách 2, 3:** kiểm tra trực tiếp định nghĩa với mọi `j ≠ i` (bitmask: `a & b == a` ⇔ mọi bit của `a` có trong `b` ⇔ mọi công ty của `i` có trong danh sách `j`).

**Cách 4:** cần chứng minh "người `i` bị loại ⇔ `i` là tập con của một người trong `kept` lúc xét `i`".
- (⇐) hiển nhiên: người trong `kept` là người khác `i`.
- (⇒) giả sử `i ⊆ j` với `j ≠ i`. Vì danh sách khác nhau nên `|j| > |i|` ⇒ `j` được xét **trước** `i`. Chọn `j` là người có kích thước **lớn nhất** trong các người chứa `i`. Nếu `j` bị loại thì `j ⊆ k` với `|k| > |j|`, và `i ⊆ k` (bắc cầu) — mâu thuẫn với tính lớn nhất. Vậy `j ∈ kept` và `i` bị phát hiện.

**Cách 5:** `cand = ∩_{c ∈ fav[i]} {người thích c}` = tập người chứa toàn bộ danh sách `i` (chính `i` luôn thuộc). `i` bị loại ⇔ tập này có người khác ⇔ `cand ≠ {i}`. Dừng sớm khi `cand == {i}` là an toàn vì AND chỉ làm tập nhỏ đi.

## 11. Phân tích Time / Space

Gọi `n` = số người (≤ 100), `m` = độ dài lớn nhất (≤ 500), `S = Σ|fav[i]|` (≤ 50 000), `U` = số công ty khác nhau (≤ S), `w` ≈ 30 (bit mỗi "chữ số" của int Python).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Brute list | `O(n²·m²)` | `O(1)` | quá chậm |
| 2. Set issubset | `O(S + n²·m)` | `O(S)` | ổn với giới hạn đề |
| 3. Bitmask công ty | `O(S + n²·U/w)` | `O(n·U/w)` | AND trên số lớn, nhanh ở tầng C |
| 4. Sắp + kept | `O(S + n log n + n·|kept|·m)` | `O(S)` | thường ít phép so |
| 5. Chỉ mục ngược | `O(S · n/w)` | `O(U · n/w)` | gần tuyến tính theo input |

(Bỏ qua chi phí băm chuỗi dài ≤ 20 ký tự — coi như hằng số.)

## 12. Bộ test

Ví dụ đề, edge case (1 người, chuỗi tập con lồng nhau, không ai giao nhau, tập con "tiền tố chuỗi" như `"goo"` vs `"google"`), đối chiếu ngẫu nhiên với brute force (seed cố định, đảm bảo các danh sách phân biệt như ràng buộc), và kết quả phải tăng dần.

In [20]:
ALL = {
    "brute": people_brute,
    "sets": people_sets,
    "bitmask": people_bitmask,
    "sorted_kept": people_sorted_kept,
    "inverted": people_inverted,
    "SolutionSets": SolutionSets().peopleIndexes,
    "SolutionBitmask": SolutionBitmask().peopleIndexes,
    "SolutionSortedKept": SolutionSortedKept().peopleIndexes,
    "Solution": Solution().peopleIndexes,
}
edge = [
    ([["a"]], [0]),
    ([["a"], ["a", "b"], ["a", "b", "c"]], [2]),
    ([["a", "b", "c"], ["a", "b"], ["a"]], [0]),
    ([["goo"], ["google"]], [0, 1]),
    ([["a", "b"], ["b", "c"], ["a", "c"]], [0, 1, 2]),
    ([["x", "y"], ["y", "z"], ["x", "y", "z"], ["w"]], [2, 3]),
]
for fav, want in examples + edge:
    for name, f in ALL.items():
        got = f(fav)
        assert got == want, (name, fav, got, want)
print("Ví dụ đề + edge case: OK cho", len(ALL), "cách")

Ví dụ đề + edge case: OK cho 9 cách


In [21]:
rng = random.Random(2024)
for _ in range(500):
    voc = list(string.ascii_lowercase[:rng.randint(1, 8)])
    n = rng.randint(1, min(12, 2 ** len(voc) - 1))
    fav = random_case(n, len(voc), voc, rng)
    want = people_brute(fav)
    assert want == sorted(want)
    for name, f in ALL.items():
        assert f(fav) == want, (name, fav)
print("500 test ngẫu nhiên nhỏ: mọi cách khớp brute force")

r = {name: f(worst) for name, f in ALL.items() if name != "brute"}
assert len({tuple(v) for v in r.values()}) == 1
print("Test cỡ giới hạn đề: các cách nhanh cho cùng kết quả,", len(r["Solution"]), "người được giữ")

500 test ngẫu nhiên nhỏ: mọi cách khớp brute force


Test cỡ giới hạn đề: các cách nhanh cho cùng kết quả, 89 người được giữ


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên loại `j == i` (so với chính mình)

Mọi tập đều là tập con của chính nó ⇒ không ai được giữ.

In [22]:
def wrong_self(fav):
    sets = [set(l) for l in fav]
    return [i for i in range(len(sets)) if not any(sets[i] <= sets[j] for j in range(len(sets)))]

print("Sai :", wrong_self(ex1), "| Đúng:", Solution().peopleIndexes(ex1))

Sai : [] | Đúng: [0, 1, 4]


**Sửa:** thêm điều kiện `i != j`.

### Lỗi 2 — Chỉ so với người đứng **sau** (`j > i`)

Thói quen "duyệt cặp `i < j`" từ các bài đối xứng; nhưng quan hệ tập con **không đối xứng**.

In [23]:
def wrong_one_direction(fav):
    sets = [set(l) for l in fav]
    n = len(sets)
    return [i for i in range(n) if not any(sets[i] <= sets[j] for j in range(i + 1, n))]

print("Sai :", wrong_one_direction(ex1), "| Đúng:", Solution().peopleIndexes(ex1))
print("Người 2 ⊆ người 0, người 3 ⊆ người 0, 1 — đều đứng trước nên bản sai không kiểm tra.")

Sai : [0, 1, 2, 3, 4] | Đúng: [0, 1, 4]
Người 2 ⊆ người 0, người 3 ⊆ người 0, 1 — đều đứng trước nên bản sai không kiểm tra.


**Sửa:** duyệt mọi `j ≠ i` (hoặc dùng cách 4 có sắp xếp).

### Lỗi 3 — Đảo chiều phép AND của bitmask

`a & b == b` kiểm tra `b ⊆ a`, không phải `a ⊆ b`.

In [24]:
def wrong_and_direction(fav):
    ids, masks = {}, []
    for lst in fav:
        m = 0
        for c in lst:
            m |= 1 << ids.setdefault(c, len(ids))
        masks.append(m)
    return [i for i, a in enumerate(masks)
            if not any(i != j and a & b == b for j, b in enumerate(masks))]   # SAI chiều

print("Sai :", wrong_and_direction(ex1), "| Đúng:", Solution().peopleIndexes(ex1))

Sai : [3, 4] | Đúng: [0, 1, 4]


**Sửa:** `a & b == a` (với `a` là mask của người đang xét).

### Lỗi 4 — Kiểm tra bằng chuỗi ghép (substring)

Ghép danh sách thành một chuỗi rồi dùng `c in chuỗi` ⇒ `"goo"` "nằm trong" `"google"`.

In [25]:
def wrong_substring(fav):
    joined = [",".join(l) for l in fav]
    n = len(fav)
    return [i for i in range(n)
            if not any(i != j and all(c in joined[j] for c in fav[i]) for j in range(n))]

case = [["goo"], ["google"]]
print("Sai :", wrong_substring(case), "| Đúng:", Solution().peopleIndexes(case))

Sai : [1] | Đúng: [0, 1]


**Sửa:** so trên **tập các chuỗi** (`set`), không phải trên chuỗi ghép.

### Lỗi 5 — Trả về theo thứ tự đã sắp kích thước

Cách 4 xét người theo kích thước giảm dần; quên `sorted(kept)` ⇒ sai thứ tự đề yêu cầu.

In [26]:
def wrong_order(fav):
    sets = [set(l) for l in fav]
    kept = []
    for i in sorted(range(len(sets)), key=lambda i: -len(sets[i])):
        if not any(sets[i] <= sets[k] for k in kept):
            kept.append(i)
    return kept                              # SAI: chưa sắp tăng dần

case = [["a"], ["b", "c"], ["d", "e", "f"]]
print("Sai :", wrong_order(case), "| Đúng:", Solution().peopleIndexes(case))

Sai : [2, 1, 0] | Đúng: [0, 1, 2]


**Sửa:** `return sorted(kept)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Là tập con của …"** ⇒ đổi sang `set` và dùng `<=` / `issubset`.
- Nhiều phép giao/tập con trên vũ trụ phần tử **rời rạc** ⇒ **đánh id + bitmask** (Python int không giới hạn bit).
- Quan hệ có **bắc cầu** và **kích thước đơn điệu** ⇒ **sắp theo kích thước** và chỉ so với "đại diện" đã giữ.
- Muốn biết "ai chứa toàn bộ X" ⇒ **chỉ mục ngược** (phần tử → tập chủ sở hữu) rồi giao lại.
- Đọc kỹ ràng buộc "danh sách khác nhau từng đôi": nó loại bỏ trường hợp hai tập bằng nhau (nếu không, phải quyết định ai được giữ).

## 15. Biến thể của bài

1. **Cho phép danh sách trùng nhau:** hai người cùng danh sách thì cả hai đều là "tập con của người khác" ⇒ phải dùng `<=` cẩn thận, cách 4 cần sửa (so cả người cùng kích thước).
2. **Tập tối tiểu:** người mà danh sách **không chứa** trọn danh sách của ai khác (bài tập 5).
3. **Với mỗi người, liệt kê mọi "siêu tập"** (bài tập 4).
4. **Đếm số cặp `(i, j)` với `fav[i] ⊂ fav[j]`** — dùng chỉ mục ngược: `popcount(cand) − 1`.
5. **Maximum Product of Word Lengths** (LeetCode 318): bitmask chữ cái để kiểm tra hai từ **không giao nhau** (`a & b == 0`).

In [27]:
def count_subset_pairs(fav):
    P = defaultdict(int)
    for i, lst in enumerate(fav):
        for c in lst:
            P[c] |= 1 << i
    total = 0
    for i, lst in enumerate(fav):
        cand = ~0
        for c in lst:
            cand &= P[c]
        total += bin(cand).count("1") - 1      # bỏ chính i
    return total

print("số cặp tập con thực sự trong ví dụ 1:", count_subset_pairs(ex1))

số cặp tập con thực sự trong ví dụ 1: 4


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute list | `all(c in list_j)` | `O(n²·m²)` | `O(1)` | Kiểm chứng `n, m` nhỏ |
| 2. Set issubset | `s_i <= s_j` mọi cặp | `O(n²·m)` | `O(S)` | Lời giải gọn, đủ nhanh |
| 3. Bitmask | `a & b == a` | `O(n²·U/w)` | `O(n·U/w)` | Nhiều phép tập con, vũ trụ vừa |
| 4. Sắp + kept | So với người lớn hơn đã giữ | `O(n·|kept|·m)` | `O(S)` | Thực tế nhanh, dễ giải thích |
| 5. Chỉ mục ngược | AND bitmask người theo công ty | `O(S·n/w)` | `O(U·n/w)` | Tối ưu theo kích thước input |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output):** `peopleIndexes([["a","b"],["b","c"],["a","b","c"],["d"]])` = ?

**Bài 2 (tính tay):** với ví dụ 1, đánh id công ty theo thứ tự xuất hiện và viết mask (dạng số nguyên) của từng người.

**Bài 3 (điền TODO):** hoàn thiện `people_todo` dùng `set.issubset`.

**Bài 4 (tự viết lại):** viết `supersets_of(fav)` trả về list, phần tử thứ `i` là danh sách (tăng dần) các `j ≠ i` có `fav[i] ⊆ fav[j]`.

**Bài 5 (mini challenge):** viết `minimal_people(fav)` trả về (tăng dần) các người mà danh sách **không chứa trọn** danh sách của bất kỳ người nào khác, hiệu quả (gợi ý: sắp kích thước **tăng dần** + kept).

In [28]:
# Bài 3 — điền TODO (chỉ định nghĩa, không gọi)
def people_todo(fav):
    sets = [set(l) for l in fav]
    res = []
    for i in range(len(sets)):
        # TODO: kiểm tra có j != i nào mà sets[i] là tập con của sets[j]
        # TODO: nếu không có thì thêm i vào res
        pass
    return res

## 18. Gợi ý 3 mức cho bài tập

**Bài 1:** (1) `{a,b}` và `{b,c}` đều ⊆ `{a,b,c}`. (2) `{d}` không ⊆ ai. (3) Kết quả 2 phần tử.

**Bài 2:** (1) id: leetcode 0, google 1, facebook 2, microsoft 3, amazon 4. (2) Người 0 = bit 0,1,2 = 7. (3) Người 1 = bit 1,3 = 10; người 2 = 6; người 3 = 2; người 4 = 16.

**Bài 3:** (1) `any(i != j and sets[i].issubset(sets[j]) for j in ...)`. (2) `if not ...: res.append(i)`. (3) `i` tăng dần nên `res` tự tăng dần.

**Bài 4:** (1) Chỉ mục ngược cho `cand`. (2) `bits(cand)` bỏ `i`. (3) Hoặc hai vòng với `set`.

**Bài 5:** (1) Đối xứng với bài gốc: đổi vai `⊆` thành `⊇`. (2) Xét theo kích thước **tăng dần**, so với người đã giữ (nhỏ hơn): `sets[k] <= sets[i]`. (3) Chứng minh bằng bắc cầu như mục 10.

## 19. Đáp án tham khảo cho bài tập

In [29]:
# Bài 1
ans1 = Solution().peopleIndexes([["a", "b"], ["b", "c"], ["a", "b", "c"], ["d"]])
print("Bài 1:", ans1)
assert ans1 == [2, 3]

# Bài 2
ids2, masks2 = {}, []
for lst in ex1:
    m = 0
    for c in lst:
        m |= 1 << ids2.setdefault(c, len(ids2))
    masks2.append(m)
print("Bài 2:", ids2, masks2)
assert masks2 == [7, 10, 6, 2, 16]

# Bài 3
def people_todo_sol(fav):
    sets = [set(l) for l in fav]
    res = []
    for i in range(len(sets)):
        if not any(i != j and sets[i].issubset(sets[j]) for j in range(len(sets))):
            res.append(i)
    return res

for fav, want in examples + edge:
    assert people_todo_sol(fav) == want
print("Bài 3: OK")

Bài 1: [2, 3]
Bài 2: {'leetcode': 0, 'google': 1, 'facebook': 2, 'microsoft': 3, 'amazon': 4} [7, 10, 6, 2, 16]
Bài 3: OK


In [30]:
# Bài 4
def supersets_of(fav):
    n = len(fav)
    P = defaultdict(int)
    for i, lst in enumerate(fav):
        for c in lst:
            P[c] |= 1 << i
    out = []
    for i, lst in enumerate(fav):
        cand = ~0
        for c in lst:
            cand &= P[c]
        out.append([j for j in bits(cand, n) if j != i])
    return out

# Bài 5
def minimal_people(fav):
    sets = [set(l) for l in fav]
    kept = []
    for i in sorted(range(len(sets)), key=lambda i: len(sets[i])):
        if not any(sets[k] <= sets[i] for k in kept):
            kept.append(i)
    return sorted(kept)

def minimal_brute(fav):
    sets = [set(l) for l in fav]
    return [i for i in range(len(sets)) if not any(i != j and sets[j] <= sets[i] for j in range(len(sets)))]

assert supersets_of(ex1) == [[], [], [0], [0, 1, 2], []]
assert minimal_people(ex1) == [3, 4]
rng = random.Random(19)
for _ in range(400):
    voc = list(string.ascii_lowercase[:rng.randint(1, 7)])
    fav = random_case(rng.randint(1, min(10, 2 ** len(voc) - 1)), len(voc), voc, rng)
    sets = [set(l) for l in fav]
    assert supersets_of(fav) == [[j for j in range(len(fav)) if j != i and sets[i] <= sets[j]] for i in range(len(fav))]
    assert minimal_people(fav) == minimal_brute(fav)
print("Bài 4:", supersets_of(ex1))
print("Bài 5:", minimal_people(ex1), "| OK (400 test ngẫu nhiên)")

Bài 4: [[], [], [0], [0, 1, 2], []]
Bài 5: [3, 4] | OK (400 test ngẫu nhiên)


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template set + sắp kích thước + kept:**

```python
sets = [set(l) for l in fav]
kept = []
for i in sorted(range(len(sets)), key=lambda i: -len(sets[i])):
    if not any(sets[i] <= sets[k] for k in kept):
        kept.append(i)
return sorted(kept)
```

**Template chỉ mục ngược (bitmask người):**

```python
P = defaultdict(int)
for i, lst in enumerate(fav):
    for c in lst: P[c] |= 1 << i
for i, lst in enumerate(fav):
    cand = ~0
    for c in lst: cand &= P[c]
    if cand == 1 << i: res.append(i)
```

**Checklist:**
- [ ] Bỏ qua `j == i`.
- [ ] So **mọi** `j ≠ i` (không chỉ `j > i`), hoặc dùng sắp xếp + chứng minh bắc cầu.
- [ ] Bitmask: `a & b == a` nghĩa là `a ⊆ b`.
- [ ] So trên tập chuỗi, không trên chuỗi ghép.
- [ ] Kết quả **tăng dần**.